In [0]:
#Create two DataFrames and register them as temp views
from pyspark.sql.functions import col, count, avg, sum, row_number
from pyspark.sql.window import Window

emp_data = [(1, "Alice", "IT", 55000), (2, "Ben", "HR", 48000), (3, "Cara", "IT", 62000)]
employees = spark.createDataFrame(emp_data, ["id", "name", "department", "salary"])
employees.createOrReplaceTempView("employees")

dept_data = [("IT",), ("HR",), ("Finance",)]
departments = spark.createDataFrame(dept_data, ["department_id"])
departments.createOrReplaceTempView("departments")

In [0]:
#Rewrite Day 3's filter in SQL
#Pyspark Version
employees.select("name","salary").filter(col("salary") >50000).display()

#SQL Version
spark.sql("SELECT name, salary FROM employees WHERE salary > 50000").display()

#Note : Confirm both return the same 2 rows.

In [0]:
#Rewrite Day 6's aggregation in SQL
employees.groupBy("department").agg(count("*").alias("emp_count"), avg("salary").alias("avg_salary")).display()

# SQL version
spark.sql("""
    SELECT department, COUNT(*) AS emp_count, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department
""").display()

In [0]:
#Rewrite Day 7's orphan-detection join in SQL
# PySpark version
employees.join(departments, employees.department == departments.department_id, "left") \
    .filter(departments.department_id.isNull()).display()

# SQL version
spark.sql("""
    SELECT e.*
    FROM employees e
    LEFT JOIN departments d ON e.department = d.department_id
    WHERE d.department_id IS NULL
""").display()

In [0]:
#Rewrite Day 8's window function in SQL
# Pyspark Version
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())
employees.withColumn("row_num", row_number().over(window_spec)).display()

## SQL version
spark.sql("""
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY department ORDER BY salary DESC
    ) AS row_num
    FROM employees
""").display()
